# Sprint Semana 7 — Traductor LSP: Expansión de Dataset

**Proyecto:** Reconocimiento de Lengua de Señas Peruana (LSP)  
**Fecha:** 2026-05-29 | **Branch:** ENTREGA_SEMANA_07  
**Nuevos datasets:** Abecedario · Keypoints/PKL · SRT · Glosas (143 señas)

---

## 1. Datasets Incorporados

| Fuente | Contenido | Volumen |
|--------|-----------|--------|
| **Abecedario** | 24 letras LSP · 150 imágenes/letra | 3,600 JPG · 17 MB |
| **Keypoints/pkl** | Keypoints precomputados · 27 viñetas | 3,684 PKL · 695 MB |
| **SRT** | Subtítulos segmentados por seña | 27 archivos · 224 KB |
| **Glosas** | 143 señas · 526 MP4 + 525 EAF | 2.8 GB |

In [ ]:
from pathlib import Path

ROOT = Path("..") if Path("../data").exists() else Path(".")
DATA = ROOT / "data"

for name, subpath in [("Abecedario", "Abecedario"), ("Keypoints", "Keypoints/pkl"), ("SRT", "SRT/SRT_SEGMENTED_SIGN"), ("Glosas", "Glosas")]:
    path = DATA / subpath
    if path.exists():
        items = list(path.iterdir())
        print(f"{name:12s}: {len(items):4d} items")
    else:
        print(f"{name:12s}: no encontrado")

## 2. Abecedario — Señas Estáticas (24 letras)

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("..") if Path("../data").exists() else Path(".")
abc_dir = ROOT / "data" / "Abecedario"

if abc_dir.exists():
    data = {l.name: len(list(l.glob("*.jpg"))) for l in sorted(abc_dir.iterdir()) if l.is_dir()}
    df = pd.DataFrame(list(data.items()), columns=["Letra", "Imagenes"])
    print(f"Total letras: {len(df)} | Total imagenes: {df['Imagenes'].sum()}")
    print(df.to_string(index=False))
else:
    print("Carpeta no encontrada")

## 3. Glosas — Vocabulario Extendido (143 señas)

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("..") if Path("../data").exists() else Path(".")
glosas_dir = ROOT / "data" / "Glosas"

if glosas_dir.exists():
    rows = [{"Glosa": g.name, "MP4": len(list(g.glob("*.mp4"))), "EAF": len(list(g.glob("*.eaf")))} for g in sorted(glosas_dir.iterdir()) if g.is_dir()]
    df = pd.DataFrame(rows)
    print(f"Total señas: {len(df)} | MP4: {df['MP4'].sum()} | EAF: {df['EAF'].sum()}")
    print(df.head(15).to_string(index=False))
else:
    print("Carpeta no encontrada")

## 4. Keypoints PKL — Precomputados por Viñeta

In [ ]:
from pathlib import Path
import pickle

ROOT = Path("..") if Path("../data").exists() else Path(".")
pkl_root = ROOT / "data" / "Keypoints" / "pkl"

if pkl_root.exists():
    vinetas = sorted(pkl_root.iterdir())
    total = sum(len(list(v.glob("*.pkl"))) for v in vinetas)
    print(f"Vinetas: {len(vinetas)} | Total PKL: {total}")
    for v in vinetas:
        n = len(list(v.glob("*.pkl")))
        print(f"  {v.name:35s}: {n} archivos")
    sample = next(vinetas[0].glob("*.pkl"), None)
    if sample:
        with open(sample, "rb") as f:
            obj = pickle.load(f)
        print(f"Muestra {sample.name}: tipo={type(obj).__name__}", end="")
        if hasattr(obj, "shape"): print(f" shape={obj.shape}")
        elif isinstance(obj, dict): print(f" keys={list(obj.keys())[:4]}")
        else: print(f" len={len(obj)}")
else:
    print("Carpeta no encontrada")

## 5. SRT — Subtítulos Segmentados

In [ ]:
from pathlib import Path
import re

ROOT = Path("..") if Path("../data").exists() else Path(".")
srt_dir = ROOT / "data" / "SRT" / "SRT_SEGMENTED_SIGN"

if srt_dir.exists():
    srts = sorted(srt_dir.glob("*.srt"))
    print(f"Archivos SRT: {len(srts)}")
    if srts:
        with open(srts[0], encoding="utf-8", errors="replace") as f:
            content = f.read()
        segs = re.findall(r"\d+\n\d{2}:\d{2}:\d{2}", content)
        print(f"Muestra: {srts[0].name} — {len(segs)} segmentos")
        print("Primeras lineas:")
        for line in content.split("\n")[:12]:
            if line.strip(): print(f"  {line}")
else:
    print("Carpeta no encontrada")

## 6. Plan de Experimentos Sprint 7

| Experimento | Input | Modelo | Meta |
|-------------|-------|--------|------|
| **A — Abecedario** | JPG → 42 features | LogReg / MLP | F1 > 0.90 |
| **B — Glosas 143 clases** | MP4 → sliding window | BiLSTM+ST-GCN | F1 > 0.55 |
| **C — PKL fine-tune** | PKL precomputados | Backbone S6 | +5pp sobre S6 |
| **D — SRT segmentación** | Timestamps SRT | Pipeline S6 | Mejor alineación |



## 7. Checklist Validación Sprint 7

| Requisito | Estado |
|-----------|--------|
| Split correcto (GroupKFold por signer) | ✅ |
| Fit solo en train | ✅ |
| Seeds fijadas (42) | ✅ |
| Datos en LFS (Abecedario + Keypoints + SRT + Glosas) | ✅ |
| Manifest MD5 verificado | ✅  |

---
**Entregable:** Entrega Sprint Semana7/ENTREGA_SPRINT_SEMANA7.md

## 8. Checklist de Validación Sprint 7

Ejecución de `scripts/calibracion_s7.py` — valida los 5 ítems del checklist sobre el dataset PKL con GroupKFold.

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "../scripts/calibracion_s7.py"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[:500])

| # | Requisito | Estado |
|---|-----------|--------|
| 1 | Split correcto — `GroupKFold(n=5, groups=vineta)` | ✅ |
| 2 | Fit solo en train — `Pipeline(StandardScaler → LogReg)` | ✅ |
| 3 | Seeds fijadas — `np.random.seed(42)`, `random_state=42` | ✅ |
| 4 | Sin cambios de data — MD5 calculado en cada ejecución | ✅ |
| 5 | Logs completos — `data/calibracion_s7_resumen.txt` + CSV | ✅ |
| 6 | Nuevos datos en LFS — Abecedario + Keypoints + SRT + Glosas | ✅ |

**Script:** `scripts/calibracion_s7.py` | **Logs:** `data/calibracion_s7_resumen.txt`, `data/calibracion_s7_ablacion.csv`